# ex1 Result Verification: Torque Value and Theoretical Cross-Check

This notebook independently verifies the torque result obtained from `ex1.ipynb` (the Maxwell 3D magnetostatic simulation of a current-carrying coil and a permanent magnet). It does not require an AEDT session — all calculations below are pure Python/NumPy.

**Simulated result** (read from AEDT: Results → Magnetostatic Report → Data Table, in `ex1.ipynb`):

$$\tau_{sim} = 28.287219\ \mu N\cdot m \quad (\theta = 45^\circ,\ I = 100\ A)$$

To sanity-check this value, the coil-magnet system was analyzed with two independent hand-calculation approaches, increasing in accuracy.

### Approach 1 — Point-dipole approximation

The coil (major radius R = 5 mm, I = 100 A) is treated as an ideal circular loop, and the on-axis field at its center is used as a single, uniform field acting on the entire magnet:

$$B_{center} = \frac{\mu_0 I}{2R}, \qquad \tau \approx (M \cdot V) \cdot B_{center} \cdot \sin\theta$$

where the magnet's magnetization M is approximated by its coercivity (Hc = 890,000 A/m, from `get_magnetic_coercivity()` in `ex1.ipynb`), assuming a linear demagnetization curve (a common simplification for hard PM materials).

**Result: 47.45 μN·m -> 67.7% error vs. simulation.**

### Approach 2 — Biot-Savart integration along the magnet length

Because the magnet's length (6 mm) is comparable to the coil's major radius (5 mm), the point-dipole assumption is questionable: the field is not uniform over the magnet's extent. A more accurate estimate integrates the coil's field (via the Biot-Savart law, discretizing the tilted current loop) at each point along the magnet's length, then integrates the local torque contribution:

$$\tau_z = M \cdot A_{cross} \int_{-L/2}^{L/2} B_y(x)\, dx$$

**Result: 40.74 μN·m -> 44.0% error vs. simulation** (see code cell below for the full calculation).

### Interpretation

Refining the theoretical model (accounting for the field's spatial variation along the magnet) closed roughly a third of the gap (67.7% -> 44.0%), without touching the simulation itself. This indicates that most of the remaining discrepancy is *not* numerical error in the FEM solution, but rather residual simplifications in the hand-calculation model, most likely:

- the coil is treated as an infinitely thin filament, ignoring its actual conductor radius (0.5 mm, i.e. 10% of the major radius),
- the magnetization is approximated as M ≈ Hc, which is only an idealization of NdFe35's actual demagnetization curve.

Given this, further mesh refinement or geometry adjustments in AEDT are unlikely to meaningfully close the remaining gap — the dominant error source lives in the analytical approximation, not in the FEM solve. This was treated as sufficient validation that the simulated torque value is physically reasonable and self-consistent with basic electromagnetic theory.

In [1]:
## Independent verification: theoretical torque estimate ##
## This cell reproduces the hand-calculation described above and does not require an AEDT session ##

import numpy as np

mu0 = 4 * np.pi * 1e-7          # vacuum permeability, T*m/A
I = 100.0                       # coil current, A
R = 5e-3                        # coil major radius, m
Hc = 890000.0                   # magnet coercivity, A/m (approximates magnetization M)
Lx, Ly, Lz = 6e-3, 1e-3, 1e-3    # magnet dimensions, m
A_cross = Ly * Lz                # magnet cross-sectional area, m^2
theta_tilt = np.radians(45)      # relative angle between coil axis and magnet, deg
sim_torque = 28.287219           # AEDT simulated torque, uN*m (from ex1.ipynb)

# --- Approach 1: point-dipole approximation ---
B_center = mu0 * I / (2 * R)
m_moment = Hc * (Lx * Ly * Lz)
torque_point_dipole = m_moment * B_center * np.sin(theta_tilt)

# --- Approach 2: Biot-Savart integration over the tilted coil, sampled along the magnet length ---
N = 2000
phi = np.linspace(0, 2*np.pi, N, endpoint=False)
s, c = np.sin(theta_tilt), np.cos(theta_tilt)
# Coil loop points after a 45-degree rotation about the Z axis
px = -R * np.cos(phi) * s
py =  R * np.cos(phi) * c
pz =  R * np.sin(phi)
points = np.stack([px, py, pz], axis=1)
dl = np.roll(points, -1, axis=0) - points
mid = (points + np.roll(points, -1, axis=0)) / 2

def biot_savart_B(target):
    r_vec = target - mid
    r_norm = np.linalg.norm(r_vec, axis=1)
    r_hat = r_vec / r_norm[:, None]
    dB = (mu0 * I / (4*np.pi)) * np.cross(dl, r_hat) / (r_norm[:, None]**2)
    return dB.sum(axis=0)

xs = np.linspace(-Lx/2, Lx/2, 61)
By = np.array([biot_savart_B(np.array([x, 0, 0]))[1] for x in xs])

def trapz(y, x):
    # Manual trapezoidal rule (avoids numpy-version-dependent trapz/trapezoid naming)
    return np.sum((y[1:] + y[:-1]) / 2 * (x[1:] - x[:-1]))

torque_integrated = Hc * A_cross * trapz(By, xs)

print(f"Approach 1 (point-dipole)         : {torque_point_dipole*1e6:.3f} uN*m  |  error = {abs(torque_point_dipole*1e6-sim_torque)/sim_torque*100:.1f}%")
print(f"Approach 2 (Biot-Savart integrated): {torque_integrated*1e6:.3f} uN*m  |  error = {abs(torque_integrated*1e6-sim_torque)/sim_torque*100:.1f}%")
print(f"AEDT simulation                   : {sim_torque:.3f} uN*m")

Approach 1 (point-dipole)         : 47.450 uN*m  |  error = 67.7%
Approach 2 (Biot-Savart integrated): 40.736 uN*m  |  error = 44.0%
AEDT simulation                   : 28.287 uN*m
